# VQE on IBM Quantum Computers using Qiskit

This notebook implements the **Variational Quantum Eigensolver (VQE)** for single-qubit systems using **Qiskit** to run on real IBM quantum hardware or simulators.

## Key Features:

1. **Real quantum hardware**: Run on IBM's quantum processors
2. **Qiskit Runtime**: Use optimized VQE primitives
3. **Noise modeling**: Simulate realistic quantum noise
4. **Error mitigation**: Techniques to improve results
5. **Comparison**: Ideal simulator vs. noisy simulator vs. real hardware

## Hamiltonian:

We'll work with a general single-qubit Hamiltonian:

$$H = h_I I + h_X X + h_Y Y + h_Z Z$$

## VQE Circuit:

Universal single-qubit ansatz: $U(\theta) = R_z(\theta_2) R_y(\theta_1) R_z(\theta_0)$

## 1. Setup and Installation

First, let's install and import the necessary packages.

In [ ]:
# Install Qiskit if needed (uncomment if not installed)
# !pip install qiskit qiskit-ibm-runtime qiskit-aer matplotlib numpy scipy

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Qiskit imports
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Operator, Pauli, SparsePauliOp, Statevector
from qiskit.circuit import Parameter, ParameterVector
from qiskit.primitives import Estimator, Sampler
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.providers.fake_provider import FakeManila, FakeManilaV2
from qiskit.visualization import plot_histogram, plot_bloch_multivector

# For IBM Quantum access (optional)
try:
    from qiskit_ibm_runtime import QiskitRuntimeService, Estimator as RuntimeEstimator
    IBM_AVAILABLE = True
except ImportError:
    IBM_AVAILABLE = False
    print("IBM Runtime not available. Install with: pip install qiskit-ibm-runtime")

print("Qiskit imports successful!")
print(f"IBM Quantum Runtime available: {IBM_AVAILABLE}")

## 2. Define the Hamiltonian

We'll create a Hamiltonian using Qiskit's `SparsePauliOp` class.

In [ ]:
def create_hamiltonian(h_I, h_X, h_Y, h_Z):
    """
    Create a single-qubit Hamiltonian using Qiskit.
    
    H = h_I * I + h_X * X + h_Y * Y + h_Z * Z
    
    Parameters:
    -----------
    h_I, h_X, h_Y, h_Z : float
        Pauli coefficients
    
    Returns:
    --------
    hamiltonian : SparsePauliOp
        The Hamiltonian operator
    """
    # Create Pauli operators
    pauli_list = [('I', h_I), ('X', h_X), ('Y', h_Y), ('Z', h_Z)]
    
    # Filter out zero coefficients
    pauli_list = [(p, c) for p, c in pauli_list if abs(c) > 1e-10]
    
    if not pauli_list:
        pauli_list = [('I', 0.0)]
    
    hamiltonian = SparsePauliOp.from_list(pauli_list)
    
    return hamiltonian

# Define Hamiltonian parameters
h_I = 1.0
h_X = 0.5
h_Y = 0.3
h_Z = 0.8

# Create Hamiltonian
hamiltonian = create_hamiltonian(h_I, h_X, h_Y, h_Z)

print("="*60)
print("HAMILTONIAN")
print("="*60)
print(f"\nH = {h_I:.3f} I + {h_X:.3f} X + {h_Y:.3f} Y + {h_Z:.3f} Z")
print(f"\nQiskit representation:")
print(hamiltonian)

# Compute exact ground state
H_matrix = hamiltonian.to_matrix()
eigenvalues, eigenvectors = np.linalg.eigh(H_matrix)
E_exact = eigenvalues[0]
psi_exact = eigenvectors[:, 0]

print(f"\nExact ground state energy: {E_exact:.10f}")
print(f"Exact excited state energy: {eigenvalues[1]:.10f}")
print(f"Energy gap: {eigenvalues[1] - E_exact:.10f}")

## 3. VQE Ansatz Circuit

We create a parameterized quantum circuit using Qiskit.

In [ ]:
def create_ansatz():
    """
    Create VQE ansatz circuit: U = Rz(θ₂) Ry(θ₁) Rz(θ₀)
    
    This is the universal single-qubit gate decomposition.
    """
    # Create parameters
    theta = ParameterVector('θ', 3)
    
    # Create circuit
    qc = QuantumCircuit(1)
    
    # Apply rotations
    qc.rz(theta[0], 0)
    qc.ry(theta[1], 0)
    qc.rz(theta[2], 0)
    
    return qc, theta

# Create ansatz
ansatz, params = create_ansatz()

print("VQE Ansatz Circuit:")
print(ansatz)
print(f"\nParameters: {params}")

# Visualize
ansatz.draw('mpl', style='iqp')
plt.title('VQE Ansatz: Universal Single-Qubit Gate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. VQE with Qiskit Primitives

Qiskit's **Estimator** primitive allows us to compute expectation values efficiently.

In [ ]:
class QiskitVQE:
    """
    VQE implementation using Qiskit primitives.
    """
    
    def __init__(self, hamiltonian, ansatz, estimator, initial_point=None):
        """
        Initialize VQE.
        
        Parameters:
        -----------
        hamiltonian : SparsePauliOp
            The Hamiltonian to minimize
        ansatz : QuantumCircuit
            Parameterized quantum circuit
        estimator : Estimator
            Qiskit Estimator primitive
        initial_point : array-like, optional
            Initial parameter values
        """
        self.hamiltonian = hamiltonian
        self.ansatz = ansatz
        self.estimator = estimator
        self.initial_point = initial_point
        
        self.energy_history = []
        self.param_history = []
        self.n_evaluations = 0
    
    def cost_function(self, params):
        """
        Compute energy expectation value for given parameters.
        """
        # Use Estimator to compute <ψ(θ)|H|ψ(θ)>
        job = self.estimator.run([self.ansatz], [self.hamiltonian], [params])
        result = job.result()
        energy = result.values[0]
        
        # Store history
        self.energy_history.append(energy)
        self.param_history.append(params.copy())
        self.n_evaluations += 1
        
        return energy
    
    def optimize(self, method='COBYLA', maxiter=100):
        """
        Run VQE optimization.
        """
        if self.initial_point is None:
            self.initial_point = np.random.uniform(0, 2*np.pi, self.ansatz.num_parameters)
        
        print(f"Starting VQE optimization...")
        print(f"  Backend: {type(self.estimator).__name__}")
        print(f"  Method: {method}")
        print(f"  Initial point: {self.initial_point}")
        
        # Reset
        self.energy_history = []
        self.param_history = []
        self.n_evaluations = 0
        
        # Optimize
        result = minimize(
            self.cost_function,
            self.initial_point,
            method=method,
            options={'maxiter': maxiter}
        )
        
        print(f"\nOptimization complete!")
        print(f"  Evaluations: {self.n_evaluations}")
        print(f"  Final energy: {result.fun:.10f}")
        print(f"  Optimal params: {result.x}")
        
        return result

print("QiskitVQE class defined.")

## 5. Run VQE on Ideal Simulator

First, let's run on a noiseless simulator to verify our implementation.

In [ ]:
print("="*70)
print("VQE ON IDEAL SIMULATOR (No Noise)")
print("="*70)

# Create ideal estimator
estimator_ideal = Estimator()

# Create VQE instance
vqe_ideal = QiskitVQE(hamiltonian, ansatz, estimator_ideal)

# Run optimization
result_ideal = vqe_ideal.optimize(method='COBYLA', maxiter=100)

# Results
E_vqe_ideal = result_ideal.fun
params_optimal = result_ideal.x

print(f"\n" + "="*70)
print("RESULTS (IDEAL SIMULATOR)")
print("="*70)
print(f"Exact ground energy:    {E_exact:.10f}")
print(f"VQE ground energy:      {E_vqe_ideal:.10f}")
print(f"Absolute error:         {abs(E_vqe_ideal - E_exact):.2e}")
print(f"Relative error:         {abs(E_vqe_ideal - E_exact)/abs(E_exact)*100:.6f}%")

### Visualize Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Energy convergence
ax1 = axes[0]
ax1.plot(vqe_ideal.energy_history, 'b-', linewidth=2, alpha=0.7, label='VQE (ideal)')
ax1.axhline(y=E_exact, color='r', linestyle='--', linewidth=2, label='Exact')
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Energy', fontsize=12)
ax1.set_title('VQE Convergence (Ideal Simulator)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Error (log scale)
ax2 = axes[1]
errors = [abs(E - E_exact) for E in vqe_ideal.energy_history]
ax2.semilogy(errors, 'g-', linewidth=2)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('|E - E_exact|', fontsize=12)
ax2.set_title('Energy Error (Log Scale)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

### Visualize Final State

In [ ]:
# Create circuit with optimal parameters
optimal_circuit = ansatz.assign_parameters(params_optimal)

# Get statevector
statevector = Statevector.from_instruction(optimal_circuit)

print("VQE Final State:")
print(f"  |ψ⟩ = {statevector.data[0]:.6f} |0⟩ + {statevector.data[1]:.6f} |1⟩")

# Visualize on Bloch sphere
fig = plot_bloch_multivector(statevector)
plt.suptitle('VQE Ground State on Bloch Sphere', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate fidelity with exact state
fidelity = abs(np.vdot(psi_exact, statevector.data))**2
print(f"\nFidelity with exact ground state: {fidelity:.10f}")

## 6. Run VQE on Noisy Simulator

Now let's simulate realistic quantum hardware using a noise model.

In [ ]:
print("="*70)
print("VQE ON NOISY SIMULATOR (Realistic Noise Model)")
print("="*70)

# Create fake backend (simulates IBM Manila)
fake_backend = FakeManila()

# Create noisy simulator
noise_model = NoiseModel.from_backend(fake_backend)
backend_noisy = AerSimulator(noise_model=noise_model)

print(f"\nNoise model from: {fake_backend.name}")
print(f"Number of qubits: {fake_backend.configuration().n_qubits}")
print(f"Basis gates: {noise_model.basis_gates}")

# Create noisy estimator
estimator_noisy = Estimator(backend=backend_noisy)

# Create VQE instance
vqe_noisy = QiskitVQE(hamiltonian, ansatz, estimator_noisy, initial_point=params_optimal)

# Run optimization (starting from ideal result)
result_noisy = vqe_noisy.optimize(method='COBYLA', maxiter=50)

# Results
E_vqe_noisy = result_noisy.fun
params_noisy = result_noisy.x

print(f"\n" + "="*70)
print("RESULTS (NOISY SIMULATOR)")
print("="*70)
print(f"Exact ground energy:    {E_exact:.10f}")
print(f"VQE (ideal):            {E_vqe_ideal:.10f}")
print(f"VQE (noisy):            {E_vqe_noisy:.10f}")
print(f"\nNoise impact:           {abs(E_vqe_noisy - E_vqe_ideal):.6f}")
print(f"Error (noisy):          {abs(E_vqe_noisy - E_exact):.6f}")

### Compare Ideal vs. Noisy

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Convergence comparison
ax1.plot(vqe_ideal.energy_history, 'b-', linewidth=2, label='Ideal simulator', alpha=0.7)
ax1.plot(vqe_noisy.energy_history, 'orange', linewidth=2, label='Noisy simulator', alpha=0.7)
ax1.axhline(y=E_exact, color='r', linestyle='--', linewidth=2, label='Exact')
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Energy', fontsize=12)
ax1.set_title('VQE Convergence: Ideal vs. Noisy', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Energy bar comparison
energies = [E_exact, E_vqe_ideal, E_vqe_noisy]
labels = ['Exact', 'VQE\n(Ideal)', 'VQE\n(Noisy)']
colors = ['red', 'blue', 'orange']

ax2.bar(labels, energies, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('Ground State Energy', fontsize=12)
ax2.set_title('Energy Comparison', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Add error annotations
for i, (label, energy) in enumerate(zip(labels[1:], energies[1:]), 1):
    error = abs(energy - E_exact)
    ax2.text(i, energy + 0.05, f'Δ={error:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 7. Measurement Analysis

Let's examine the individual Pauli measurements using Qiskit.

In [ ]:
def measure_pauli_terms(circuit, estimator):
    """
    Measure individual Pauli expectation values.
    """
    pauli_ops = {
        'I': SparsePauliOp.from_list([('I', 1.0)]),
        'X': SparsePauliOp.from_list([('X', 1.0)]),
        'Y': SparsePauliOp.from_list([('Y', 1.0)]),
        'Z': SparsePauliOp.from_list([('Z', 1.0)])
    }
    
    results = {}
    
    for name, op in pauli_ops.items():
        job = estimator.run([circuit], [op])
        result = job.result()
        results[name] = result.values[0]
    
    return results

# Measure on ideal simulator
circuit_optimal = ansatz.assign_parameters(params_optimal)
measurements_ideal = measure_pauli_terms(circuit_optimal, estimator_ideal)

# Measure on noisy simulator
circuit_noisy = ansatz.assign_parameters(params_noisy)
measurements_noisy = measure_pauli_terms(circuit_noisy, estimator_noisy)

# Display results
print("Pauli Expectation Values:\n")
print(f"{'Pauli':>5s} | {'Coefficient':>12s} | {'Ideal':>12s} | {'Noisy':>12s} | {'Contrib (Ideal)':>16s} | {'Contrib (Noisy)':>16s}")
print("-" * 90)

coeffs = {'I': h_I, 'X': h_X, 'Y': h_Y, 'Z': h_Z}
total_ideal = 0
total_noisy = 0

for pauli in ['I', 'X', 'Y', 'Z']:
    coeff = coeffs[pauli]
    exp_ideal = measurements_ideal[pauli]
    exp_noisy = measurements_noisy[pauli]
    contrib_ideal = coeff * exp_ideal
    contrib_noisy = coeff * exp_noisy
    total_ideal += contrib_ideal
    total_noisy += contrib_noisy
    
    print(f"{pauli:>5s} | {coeff:+12.6f} | {exp_ideal:+12.6f} | {exp_noisy:+12.6f} | {contrib_ideal:+16.6f} | {contrib_noisy:+16.6f}")

print("-" * 90)
print(f"{'Total':>5s} | {'':>12s} | {'':>12s} | {'':>12s} | {total_ideal:+16.6f} | {total_noisy:+16.6f}")
print(f"\nExact: {E_exact:+16.6f}")

In [ ]:
# Visualize Pauli measurements
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Expectation values
pauli_names = ['I', 'X', 'Y', 'Z']
ideal_vals = [measurements_ideal[p] for p in pauli_names]
noisy_vals = [measurements_noisy[p] for p in pauli_names]

x = np.arange(len(pauli_names))
width = 0.35

ax1.bar(x - width/2, ideal_vals, width, label='Ideal', alpha=0.7)
ax1.bar(x + width/2, noisy_vals, width, label='Noisy', alpha=0.7)
ax1.set_ylabel('Expectation Value', fontsize=12)
ax1.set_title('Pauli Expectation Values', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(pauli_names)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Energy contributions
ideal_contribs = [coeffs[p] * measurements_ideal[p] for p in pauli_names]
noisy_contribs = [coeffs[p] * measurements_noisy[p] for p in pauli_names]

ax2.bar(x - width/2, ideal_contribs, width, label='Ideal', alpha=0.7)
ax2.bar(x + width/2, noisy_contribs, width, label='Noisy', alpha=0.7)
ax2.set_ylabel('Energy Contribution', fontsize=12)
ax2.set_title('Pauli Contributions to Energy', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(pauli_names)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 8. Measurement Basis Circuits

Show how individual Pauli measurements are performed.

In [ ]:
def create_measurement_circuit(ansatz, params, basis='Z'):
    """
    Create circuit with measurement in specified Pauli basis.
    
    Parameters:
    -----------
    ansatz : QuantumCircuit
        Parameterized ansatz
    params : array-like
        Parameter values
    basis : str
        'X', 'Y', or 'Z'
    """
    qc = ansatz.assign_parameters(params)
    
    # Add basis rotation
    if basis == 'X':
        qc.h(0)  # Hadamard rotates Z → X
    elif basis == 'Y':
        qc.sdg(0)  # S† rotates Z → Y
        qc.h(0)
    # Z basis: no rotation needed
    
    # Add measurement
    qc.measure_all()
    
    return qc

# Create measurement circuits
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, basis in enumerate(['X', 'Y', 'Z']):
    meas_circuit = create_measurement_circuit(ansatz, params_optimal, basis)
    meas_circuit.draw('mpl', ax=axes[i], style='iqp')
    axes[i].set_title(f'Measurement in {basis} Basis', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Measurement procedure:")
print("  X basis: Apply H gate before measurement")
print("  Y basis: Apply S†H gates before measurement")
print("  Z basis: Direct measurement (computational basis)")

## 9. Shot-Based Sampling

Demonstrate the effect of finite measurement shots.

In [ ]:
from qiskit.primitives import BackendEstimator

def run_vqe_with_shots(shots_list):
    """
    Run VQE with different numbers of measurement shots.
    """
    results = []
    
    for shots in shots_list:
        print(f"\nRunning with {shots} shots...")
        
        # Create backend with specified shots
        backend = AerSimulator()
        estimator = BackendEstimator(backend=backend, options={'shots': shots})
        
        # Run VQE
        vqe = QiskitVQE(hamiltonian, ansatz, estimator, initial_point=params_optimal)
        result = vqe.optimize(method='COBYLA', maxiter=30)
        
        results.append({
            'shots': shots,
            'energy': result.fun,
            'error': abs(result.fun - E_exact),
            'history': vqe.energy_history
        })
        
        print(f"  Final energy: {result.fun:.6f}, Error: {abs(result.fun - E_exact):.6f}")
    
    return results

# Test different shot counts
shots_list = [100, 500, 1000, 5000, 10000]
shot_results = run_vqe_with_shots(shots_list)

print("\n" + "="*60)
print("SHOT BUDGET ANALYSIS")
print("="*60)
print(f"{'Shots':>8s} | {'Energy':>14s} | {'Error':>12s}")
print("-"*40)
for res in shot_results:
    print(f"{res['shots']:8d} | {res['energy']:+14.10f} | {res['error']:12.6f}")
print(f"{'Exact':>8s} | {E_exact:+14.10f} | {0.0:12.6f}")

In [ ]:
# Visualize shot dependence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Convergence for different shots
for res in shot_results:
    ax1.plot(res['history'], alpha=0.7, linewidth=2, label=f"{res['shots']} shots")
ax1.axhline(y=E_exact, color='red', linestyle='--', linewidth=2, label='Exact')
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Energy', fontsize=12)
ax1.set_title('VQE Convergence vs. Shot Budget', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Error vs. shots
shots_vals = [res['shots'] for res in shot_results]
error_vals = [res['error'] for res in shot_results]

ax2.loglog(shots_vals, error_vals, 'o-', linewidth=2, markersize=10)
ax2.set_xlabel('Number of Shots', fontsize=12)
ax2.set_ylabel('Energy Error', fontsize=12)
ax2.set_title('Error vs. Measurement Shots (Log-Log)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

# Add 1/sqrt(N) reference line
shots_ref = np.array(shots_vals)
error_ref = error_vals[0] * np.sqrt(shots_vals[0] / shots_ref)
ax2.plot(shots_ref, error_ref, 'r--', linewidth=1.5, alpha=0.5, label=r'$\propto 1/\sqrt{N}$')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\nObservation: Error decreases as ~ 1/√N (shot noise)")

## 10. Running on Real IBM Quantum Hardware (Optional)

**Note**: This requires an IBM Quantum account and access to quantum processors.

In [ ]:
if IBM_AVAILABLE:
    print("IBM Quantum Runtime is available!\n")
    print("To run on real hardware:")
    print("1. Get an IBM Quantum account at https://quantum.ibm.com/")
    print("2. Save your API token")
    print("3. Uncomment and run the code below\n")
    
    print("Example code to run on IBM hardware:")
    print("""
# Save your account (first time only)
# QiskitRuntimeService.save_account(channel='ibm_quantum', token='YOUR_TOKEN_HERE')

# Load service
service = QiskitRuntimeService()

# Get least busy backend
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=1)
print(f"Using backend: {backend.name}")

# Create Runtime Estimator
estimator_runtime = RuntimeEstimator(backend=backend)

# Run VQE
vqe_hardware = QiskitVQE(hamiltonian, ansatz, estimator_runtime, initial_point=params_optimal)
result_hardware = vqe_hardware.optimize(method='COBYLA', maxiter=20)

print(f"Hardware result: {result_hardware.fun:.6f}")
print(f"Error: {abs(result_hardware.fun - E_exact):.6f}")
    """)
else:
    print("IBM Quantum Runtime not available.")
    print("Install with: pip install qiskit-ibm-runtime")

## 11. Summary and Comparison

Let's compare all the different approaches.

In [ ]:
# Summary table
print("\n" + "="*80)
print("VQE RESULTS SUMMARY")
print("="*80)
print(f"\n{'Method':>30s} | {'Energy':>16s} | {'Error':>14s} | {'Rel. Error (%)':>16s}")
print("-"*80)

methods = [
    ("Exact Diagonalization", E_exact, 0.0),
    ("VQE (Ideal Simulator)", E_vqe_ideal, abs(E_vqe_ideal - E_exact)),
    ("VQE (Noisy Simulator)", E_vqe_noisy, abs(E_vqe_noisy - E_exact)),
]

for method, energy, error in methods:
    rel_error = (error / abs(E_exact) * 100) if E_exact != 0 else 0
    print(f"{method:>30s} | {energy:+16.10f} | {error:14.6e} | {rel_error:16.8f}")

print("\n" + "="*80)
print("KEY OBSERVATIONS")
print("="*80)
print("1. Ideal simulator achieves near-exact results (limited by optimizer)")
print("2. Noisy simulator shows realistic quantum hardware effects")
print("3. Increasing measurement shots reduces statistical error")
print("4. Real hardware would show additional systematic errors")
print("="*80)

In [ ]:
# Final comparison plot
fig = plt.figure(figsize=(14, 6))

# Plot 1: Energy comparison
ax1 = plt.subplot(121)
methods_plot = ['Exact', 'VQE\n(Ideal)', 'VQE\n(Noisy)']
energies_plot = [E_exact, E_vqe_ideal, E_vqe_noisy]
colors_plot = ['red', 'blue', 'orange']

bars = ax1.bar(methods_plot, energies_plot, color=colors_plot, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('Ground State Energy', fontsize=12)
ax1.set_title('VQE Results: Qiskit Implementation', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, energy in zip(bars, energies_plot):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{energy:.6f}', ha='center', va='bottom', fontsize=10)

# Plot 2: Error comparison (log scale)
ax2 = plt.subplot(122)
errors_plot = [abs(E_vqe_ideal - E_exact), abs(E_vqe_noisy - E_exact)]
method_labels = ['Ideal', 'Noisy']

bars2 = ax2.bar(method_labels, errors_plot, color=['blue', 'orange'], alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('Absolute Error', fontsize=12)
ax2.set_yscale('log')
ax2.set_title('Energy Error (Log Scale)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

# Add value labels
for bar, error in zip(bars2, errors_plot):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height*1.5,
             f'{error:.2e}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()